# LexAI — Notebook 03: RAG Pipeline (Legal Retrieval)

Demonstrates the Retrieval-Augmented Generation pipeline:
- **Embedding** legal documents with sentence-transformers
- **FAISS vector index** construction and search
- **Semantic search** for IPC sections and case precedents
- **Full RAG Q&A** with Groq LLM

Prerequisites: Notebook 01 (legal database built).

In [ ]:
import sys, pathlib, json, time
sys.path.insert(0, str(pathlib.Path.cwd().parent))
print('LexAI RAG Pipeline Demo')

## 1. Load Legal Documents and Build Embeddings

In [ ]:
from rag.legal_embedder import LegalEmbedder

embedder = LegalEmbedder()
documents = embedder.load_legal_documents()

print(f'Total legal documents loaded: {len(documents)}')
print('\nDocument sources breakdown:')

from collections import Counter
source_counts = Counter(d['source'] for d in documents)
for source, count in sorted(source_counts.items()):
    print(f'  {source}: {count} documents')

In [ ]:
t0 = time.time()
texts = [embedder._item_to_text(d) for d in documents]
embeddings = embedder.embed(texts)
print(f'Embedding shape: {embeddings.shape}')
print(f'Time: {time.time()-t0:.2f}s')
print(f'Embedding dim: {embeddings.shape[1]}')

## 2. Build FAISS Index

In [ ]:
from rag.vector_store import LegalVectorStore

vs = LegalVectorStore(dim=embeddings.shape[1])
vs.build_index(embeddings, documents)
print(f'Index built: {vs.index.ntotal if hasattr(vs.index, "ntotal") else len(documents)} vectors')

## 3. Semantic Search — IPC Sections

In [ ]:
queries = [
    'cheating and fraud deception financial',
    'murder culpable homicide death penalty',
    'domestic violence cruelty wife dowry',
    'rape sexual assault consent',
]

for query in queries:
    print(f'Query: "{query}"')
    q_emb = embedder.embed([query])
    results = vs.search(q_emb, k=3)
    for r in results:
        if r['source'] == 'ipc_sections':
            print(f'  [{r["score"]:.3f}] IPC {r["section"]} — {r["title"]}')
    print()

## 4. Semantic Search — Case Precedents

In [ ]:
from rag.legal_retriever import LegalRAGRetriever

retriever = LegalRAGRetriever()
retriever.build_index()

print('Searching precedents for: "rarest of rare doctrine capital punishment"')
precedents = retriever.search_precedents('rarest of rare doctrine capital punishment', k=3)
for p in precedents:
    print(f'  [{p["score"]:.3f}] {p.get("case_name", "Unknown")} ({p.get("year", "")})')
    print(f'    {p.get("legal_principle", "")[:100]}...')
    print()

## 5. Full Legal Context Retrieval

In [ ]:
context = retriever.get_legal_context(
    case_type='Criminal',
    charges=['IPC 420', 'IPC 468'],
    facts='accused cheated victim of Rs. 45 lakhs through fake investment'
)

print('Legal context keys:', list(context.keys()))
print('\nRelevant IPC sections:')
for s in context.get('relevant_sections', [])[:3]:
    print(f'  IPC {s.get("section", "?")} — {s.get("title", "")} (score: {s.get("score", 0):.3f})')

print('\nRelevant precedents:')
for p in context.get('relevant_precedents', [])[:3]:
    print(f'  {p.get("case_name", "?")} (score: {p.get("score", 0):.3f})')

## 6. RAG Q&A — Ask a Legal Question

In [ ]:
question = 'What are the key elements to prove fraud under IPC Section 420?'
print(f'Question: {question}\n')

answer = retriever.ask_legal_question(question, context_type='sections')
print('Answer:')
print(answer.get('answer', 'No answer generated'))
print('\nSources used:')
for s in answer.get('sources', [])[:3]:
    print(f'  - {s}')

## 7. Embedding Similarity Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

query_texts = [
    'fraud cheating financial crime',
    'murder homicide death',
    'property dispute civil court',
    'bail application criminal procedure',
]

q_embeddings = embedder.embed(query_texts)

# Cosine similarity matrix
norms = np.linalg.norm(q_embeddings, axis=1, keepdims=True)
normalized = q_embeddings / (norms + 1e-8)
sim_matrix = normalized @ normalized.T

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im)
ax.set_xticks(range(len(query_texts)))
ax.set_yticks(range(len(query_texts)))
labels = ['Fraud', 'Murder', 'Property', 'Bail']
ax.set_xticklabels(labels, rotation=30)
ax.set_yticklabels(labels)
ax.set_title('Query Embedding Cosine Similarity')
for i in range(len(query_texts)):
    for j in range(len(query_texts)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## Summary

The RAG pipeline:
1. **Embeds** all legal documents with sentence-transformers (dim=384)
2. **Indexes** them in FAISS IndexFlatIP for fast inner-product search
3. **Retrieves** relevant sections and precedents by semantic similarity
4. **Augments** LLM prompts with retrieved context for grounded answers

Next: See Notebook 04 for AI argument generation using this retrieved context.